# ***Approach 1 (Predict type with LightGBM)***

In [ ]:
!pip -q install -U lightgbm imbalanced-learn scikit-learn

import re
import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score

import lightgbm as lgb

RANDOM_STATE = 42


In [ ]:
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/IOT/Dataset/windows7_dataset.csv"  # <-- change if needed
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Loaded:", df.shape)
df.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: (28367, 135)


,ts,Processor(_Total) DPC Rate,Processor(_Total) pct_ Idle Time,Processor(_Total) pct_ C3 Time,Processor(_Total) pct_ Interrupt Time,Processor(_Total) pct_ C2 Time,Processor(_Total) pct_ User Time,Processor(_Total) pct_ C1 Time,Processor(_Total) pct_ Processor Time,Processor(_Total) C1 Transitions sec,...,Memory Available MBytes,Memory Modified Page List Bytes,Memory Cache Faults sec,Memory Committed Bytes,Memory System Driver Total Bytes,Memory Pages Input sec,Memory Pool Paged Resident Bytes,Memory Write Copies sec,label,type
0,1554185566,0,90.20833333,0,0.208333333,0,2.083333333,88.00928867,9.791666667,216.6151809,...,414,26374144,31.95524227,926330880,5849088,1957.675542,48304128,0,0,normal
1,1554185581,0,99.79166667,0,0,0,0.104166667,98.92048733,0.208333333,67.86145672,...,418,27045888,1.199907879,924794880,5849088,0.133323098,48107520,0,0,normal
2,1554185596,0,99.79166667,0,0,0,0.208333333,99.17886667,0.208333333,65.9999186,...,419,27353088,0.133333169,925319168,5849088,0,48033792,0,0,normal
3,1554185611,0,99.6875,0,0,0,0.104166667,99.218316,0.3125,67.39927613,...,425,27533312,0.133331901,916725760,5849088,1.066655211,47960064,0.133331901,0,normal
4,1554185626,0,99.79166667,0,0,0,0.208333333,99.24871267,0.208333333,65.53277281,...,426,27688960,0.133332193,916025344,5849088,0,47943680,0,0,normal


In [ ]:
def clean_column_name(col: str) -> str:
    col = re.sub(r"[^A-Za-z0-9_]", "_", str(col))
    col = re.sub(r"_+", "_", col).strip("_")
    return col

df = df.copy()
df.columns = [clean_column_name(c) for c in df.columns]

# Drop irrelevant columns if present
cols_to_drop = ["timestamp", "id", "label_binary"]
df = df.drop([c for c in cols_to_drop if c in df.columns], axis=1)

def time_split_df(df, time_col="ts", test_size=0.2):
    """Split by time: earliest (1-test_size) for train, latest test_size for test."""
    if time_col not in df.columns:
        raise ValueError(f"time_col='{time_col}' not found for time split.")
    df_sorted = df.sort_values(time_col).reset_index(drop=True)
    cut = int(len(df_sorted) * (1 - test_size))
    train_df = df_sorted.iloc[:cut].copy()
    test_df  = df_sorted.iloc[cut:].copy()
    return train_df, test_df

def stratified_time_split(df, y_col="type", time_col="ts", test_size=0.2, min_test=1):
    parts_train = []
    parts_test = []

    for cls, g in df.groupby(y_col):
        g = g.sort_values(time_col).reset_index(drop=True)
        n = len(g)
        if n < 2:
            # can't split a class with <2 rows
            parts_train.append(g)
            continue

        test_n = int(round(n * test_size))
        test_n = max(min_test, test_n)
        test_n = min(test_n, n - 1)  # leave at least 1 in train

        parts_train.append(g.iloc[:-test_n])
        parts_test.append(g.iloc[-test_n:])

    train_df = pd.concat(parts_train, axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    test_df  = pd.concat(parts_test, axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    return train_df, test_df

def coerce_object_to_numeric(X: pd.DataFrame) -> pd.DataFrame:
    """Convert object columns to numeric (strip %, commas, blanks)."""
    X = X.copy()
    for col in X.columns:
        if X[col].dtype == "object":
            s = (X[col].astype(str)
                    .str.replace("%", "", regex=False)
                    .str.replace(",", "", regex=False)
                    .str.strip()
                    .replace(["nan", "None", "?", ""], np.nan))
            X[col] = pd.to_numeric(s, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


In [ ]:
TARGET_COL = "type"
TIME_COL   = "ts"     # used only for time-split; can exist even if you drop it from features
USE_TIME_SPLIT = True # True recommended; set False to use random stratified split
DROP_TS_FROM_FEATURES = True  # recommended

assert TARGET_COL in df.columns, f"{TARGET_COL} not found."
assert "label" in df.columns, "Expected 'label' column to exist in this dataset."

# --- Features: drop target + leakage columns ---
drop_cols = [TARGET_COL, "label"]  # IMPORTANT: drop label when predicting type
X_all = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()
y_all = df[TARGET_COL].copy()

# Optional: drop ts from features (recommended)
if DROP_TS_FROM_FEATURES and TIME_COL in X_all.columns:
    X_all = X_all.drop(columns=[TIME_COL])

# Encode y
le = LabelEncoder()
y_enc = le.fit_transform(y_all.astype(str))
class_names = le.classes_

print("Classes:", list(class_names))
print("X shape:", X_all.shape)

# Split
if USE_TIME_SPLIT:
    train_df, test_df = stratified_time_split(df, y_col="type", time_col="ts", test_size=0.2)

    X_train = train_df.drop(columns=["type", "label"], errors="ignore")
    X_test  = test_df.drop(columns=["type", "label"], errors="ignore")

    # optionally drop ts from features (recommended)
    if DROP_TS_FROM_FEATURES and "ts" in X_train.columns:
        X_train = X_train.drop(columns=["ts"])
        X_test  = X_test.drop(columns=["ts"])

    y_train = le.transform(train_df["type"].astype(str))
    y_test  = le.transform(test_df["type"].astype(str))


print("Train:", X_train.shape, "Test:", X_test.shape)


Classes: ['backdoor', 'ddos', 'injection', 'normal', 'password', 'ransomware', 'scanning', 'xss']
X shape: (28367, 132)
Train: (22694, 132) Test: (5673, 132)


In [ ]:
pipe = Pipeline(steps=[
    ("to_numeric", FunctionTransformer(coerce_object_to_numeric, validate=False)),
    ("imputer", SimpleImputer(strategy="median")),  # fit on TRAIN only
    ("model", lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=120,
        class_weight="balanced",
        objective="multiclass",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("Accuracy:", acc)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)
print("\nClassification report:\n")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.28009871320289087
Macro F1: 0.49108320937219896
Weighted F1: 0.3365118138478714

Classification report:

              precision    recall  f1-score   support

    backdoor       0.05      0.53      0.09       356
        ddos       1.00      1.00      1.00       427
   injection       1.00      0.95      0.98       200
      normal       0.69      0.16      0.26      4477
    password       1.00      0.25      0.40       151
  ransomware       1.00      1.00      1.00        16
    scanning       1.00      0.11      0.20        45
         xss       0.00      0.00      0.00         1

    accuracy                           0.28      5673
   macro avg       0.72      0.50      0.49      5673
weighted avg       0.69      0.28      0.34      5673



# ***Approach 2 (Predict label with Decision Tree)***

In [ ]:
import re
import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

RANDOM_STATE = 42


In [ ]:
# If df is already in memory from Approach 1, you can skip this cell.

drive.mount("/content/drive")
DATA_PATH = "/content/drive/MyDrive/IOT/Dataset/windows7_dataset.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

def clean_column_name(col: str) -> str:
    col = re.sub(r"[^A-Za-z0-9_]", "_", str(col))
    col = re.sub(r"_+", "_", col).strip("_")
    return col

df = df.copy()
df.columns = [clean_column_name(c) for c in df.columns]

cols_to_drop = ["timestamp", "id", "label_binary"]
df = df.drop([c for c in cols_to_drop if c in df.columns], axis=1)

def time_split_df(df, time_col="ts", test_size=0.2):
    df_sorted = df.sort_values(time_col).reset_index(drop=True)
    cut = int(len(df_sorted) * (1 - test_size))
    return df_sorted.iloc[:cut].copy(), df_sorted.iloc[cut:].copy()

def coerce_object_to_numeric(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    for col in X.columns:
        if X[col].dtype == "object":
            s = (X[col].astype(str)
                    .str.replace("%", "", regex=False)
                    .str.replace(",", "", regex=False)
                    .str.strip()
                    .replace(["nan", "None", "?", ""], np.nan))
            X[col] = pd.to_numeric(s, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    return X

print("Loaded:", df.shape)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: (28367, 135)


In [ ]:
TARGET_COL = "label"
TIME_COL   = "ts"
USE_TIME_SPLIT = True       # True recommended; False for random stratified
DROP_TS_FROM_FEATURES = True

assert TARGET_COL in df.columns, f"{TARGET_COL} not found."
assert "type" in df.columns, "Expected 'type' column to exist in this dataset."

# IMPORTANT: drop type when predicting label (avoid leakage)
drop_cols = [TARGET_COL, "type"]
X_all = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()
y_all = df[TARGET_COL].copy().astype(int)

if DROP_TS_FROM_FEATURES and TIME_COL in X_all.columns:
    X_all = X_all.drop(columns=[TIME_COL])

if USE_TIME_SPLIT:
    train_df, test_df = time_split_df(df, time_col=TIME_COL, test_size=0.2)

    X_train = train_df.drop(columns=[TARGET_COL, "type"], errors="ignore")
    X_test  = test_df.drop(columns=[TARGET_COL, "type"], errors="ignore")

    if DROP_TS_FROM_FEATURES and TIME_COL in X_train.columns:
        X_train = X_train.drop(columns=[TIME_COL])
        X_test  = X_test.drop(columns=[TIME_COL])

    y_train = train_df[TARGET_COL].astype(int).values
    y_test  = test_df[TARGET_COL].astype(int).values
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all
    )

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Label distribution (test):", pd.Series(y_test).value_counts().to_dict())


Train: (22693, 132) Test: (5674, 132)
Label distribution (test): {0: 4663, 1: 1011}


In [ ]:
pipe = Pipeline(steps=[
    ("to_numeric", FunctionTransformer(coerce_object_to_numeric, validate=False)),
    ("imputer", SimpleImputer(strategy="median")),  # fit on TRAIN only
    ("model", DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced"  # helps when classes are imbalanced
    ))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("Accuracy:", acc)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)
print("\nClassification report:\n")
print(classification_report(y_test, y_pred, zero_division=0))


Accuracy: 0.8359182234755023
Macro F1: 0.5980158664630402
Weighted F1: 0.7970581133838905

Classification report:

              precision    recall  f1-score   support

           0       0.85      0.98      0.91      4663
           1       0.63      0.19      0.29      1011

    accuracy                           0.84      5674
   macro avg       0.74      0.58      0.60      5674
weighted avg       0.81      0.84      0.80      5674



# ***Approach 3 (Binary label + GridSearchCV, no leakage)***

In [ ]:
!pip -q install -U scikit-learn

import re
import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report, f1_score, balanced_accuracy_score

RANDOM_STATE = 42


In [ ]:
drive.mount("/content/drive")
DATA_PATH = "/content/drive/MyDrive/IOT/Dataset/windows7_dataset.csv"  # change if needed

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded:", df.shape)
df.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: (28367, 135)


,ts,Processor(_Total) DPC Rate,Processor(_Total) pct_ Idle Time,Processor(_Total) pct_ C3 Time,Processor(_Total) pct_ Interrupt Time,Processor(_Total) pct_ C2 Time,Processor(_Total) pct_ User Time,Processor(_Total) pct_ C1 Time,Processor(_Total) pct_ Processor Time,Processor(_Total) C1 Transitions sec,...,Memory Available MBytes,Memory Modified Page List Bytes,Memory Cache Faults sec,Memory Committed Bytes,Memory System Driver Total Bytes,Memory Pages Input sec,Memory Pool Paged Resident Bytes,Memory Write Copies sec,label,type
0,1554185566,0,90.20833333,0,0.208333333,0,2.083333333,88.00928867,9.791666667,216.6151809,...,414,26374144,31.95524227,926330880,5849088,1957.675542,48304128,0,0,normal
1,1554185581,0,99.79166667,0,0,0,0.104166667,98.92048733,0.208333333,67.86145672,...,418,27045888,1.199907879,924794880,5849088,0.133323098,48107520,0,0,normal
2,1554185596,0,99.79166667,0,0,0,0.208333333,99.17886667,0.208333333,65.9999186,...,419,27353088,0.133333169,925319168,5849088,0,48033792,0,0,normal
3,1554185611,0,99.6875,0,0,0,0.104166667,99.218316,0.3125,67.39927613,...,425,27533312,0.133331901,916725760,5849088,1.066655211,47960064,0.133331901,0,normal
4,1554185626,0,99.79166667,0,0,0,0.208333333,99.24871267,0.208333333,65.53277281,...,426,27688960,0.133332193,916025344,5849088,0,47943680,0,0,normal


In [ ]:
def clean_column_name(col: str) -> str:
    col = re.sub(r"[^A-Za-z0-9_]", "_", str(col))
    col = re.sub(r"_+", "_", col).strip("_")
    return col

df = df.copy()
df.columns = [clean_column_name(c) for c in df.columns]

# Drop irrelevant cols if present
cols_to_drop = ["timestamp", "id", "label_binary"]
df = df.drop([c for c in cols_to_drop if c in df.columns], axis=1)

assert "label" in df.columns, "label column not found"
assert "ts" in df.columns, "ts column not found (needed for time split)"

# IMPORTANT: predicting label => drop type from features (avoid leakage)
y = df["label"].astype(int).values
X = df.drop(columns=["label", "type"], errors="ignore").copy()

# Use time for splitting ONLY (do not use it as a feature)
X_feat = X.drop(columns=["ts"], errors="ignore").copy()

# Time-based split: earliest 80% train, latest 20% test
order = np.argsort(df["ts"].values)
X_feat = X_feat.iloc[order].reset_index(drop=True)
y = y[order]

cut = int(len(X_feat) * 0.8)
X_train, X_test = X_feat.iloc[:cut], X_feat.iloc[cut:]
y_train, y_test = y[:cut], y[cut:]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Test label distribution:", pd.Series(y_test).value_counts().to_dict())


Train: (22693, 132) Test: (5674, 132)
Test label distribution: {0: 4663, 1: 1011}


In [ ]:
def coerce_object_to_numeric(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    for col in X.columns:
        if X[col].dtype == "object":
            s = (X[col].astype(str)
                 .str.replace("%", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .str.strip()
                 .replace(["nan", "None", "?", ""], np.nan))
            X[col] = pd.to_numeric(s, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)

    # Drop columns that are entirely NaN after coercion
    all_nan_cols = [c for c in X.columns if X[c].isna().all()]
    if all_nan_cols:
        X = X.drop(columns=all_nan_cols)

    return X

pipe = Pipeline(steps=[
    ("to_numeric", FunctionTransformer(coerce_object_to_numeric, validate=False)),
    ("imputer", SimpleImputer(strategy="median")),   # fit on TRAIN only
    ("clf", DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

param_grid = {
    "clf__max_depth": [5, 10, 20, None],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 5],
    "clf__criterion": ["gini", "entropy"],
}

# Time-aware CV on training set (already time-ordered)
tscv = TimeSeriesSplit(n_splits=5)

# Use a metric that respects imbalance better than plain accuracy
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=tscv,
    scoring="f1",      # for label=1 detection; you can try "balanced_accuracy" too
    n_jobs=-1,
    refit=True
)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV score (f1):", grid.best_score_)

y_pred = grid.predict(X_test)

print("\nHoldout Accuracy:", accuracy_score(y_test, y_pred))
print("Holdout Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Holdout F1 (positive=1):", f1_score(y_test, y_pred, pos_label=1, zero_division=0))

print("\nClassification report:\n")
print(classification_report(y_test, y_pred, zero_division=0))


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Best params: {'clf__criterion': 'gini', 'clf__max_depth': 5, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2}
Best CV score (f1): nan

Holdout Accuracy: 0.8121254846669017
Holdout Balanced Accuracy: 0.8833719075161429
Holdout F1 (positive=1): 0.653446033810143

Classification report:

              precision    recall  f1-score   support

           0       1.00      0.77      0.87      4663
           1       0.49      0.99      0.65      1011

    accuracy                           0.81      5674
   macro avg       0.74      0.88      0.76      5674
weighted avg       0.91      0.81      0.83      5674

